In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver 

In [2]:
llm = ChatOllama(model = "llama3")

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [9]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [10]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy?\n\nBecause it was feeling a little crusty and wanted to work through some topping issues!',
 'explanation': 'The joke is a play on words, using puns and clever language to create humor. Here\'s a breakdown of why it works:\n\n* "Why did the pizza go to therapy?" - The setup for the joke asks about a seemingly ordinary pizza going to therapy, which is an unusual and unexpected scenario.\n* "Because..." - The phrase "because" sets up the punchline as an explanation or reason for the pizza\'s actions.\n* "...it was feeling a little crusty" - The phrase "a little crusty" has a double meaning. Crusty can refer to the crispy, flaky exterior of a pizza, but it also means being gruff or irritable (e.g., "I\'m feeling a little crusty today"). This is where the wordplay starts.\n* "...and wanted to work through some topping issues!" - The punchline reveals that the pizza\'s motivation for going to therapy is not just about its crustin

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy?\n\nBecause it was feeling a little crusty and wanted to work through some topping issues!', 'explanation': 'The joke is a play on words, using puns and clever language to create humor. Here\'s a breakdown of why it works:\n\n* "Why did the pizza go to therapy?" - The setup for the joke asks about a seemingly ordinary pizza going to therapy, which is an unusual and unexpected scenario.\n* "Because..." - The phrase "because" sets up the punchline as an explanation or reason for the pizza\'s actions.\n* "...it was feeling a little crusty" - The phrase "a little crusty" has a double meaning. Crusty can refer to the crispy, flaky exterior of a pizza, but it also means being gruff or irritable (e.g., "I\'m feeling a little crusty today"). This is where the wordplay starts.\n* "...and wanted to work through some topping issues!" - The punchline reveals that the pizza\'s motivation for going to therapy is not jus

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy?\n\nBecause it was feeling a little crusty and wanted to work through some topping issues!', 'explanation': 'The joke is a play on words, using puns and clever language to create humor. Here\'s a breakdown of why it works:\n\n* "Why did the pizza go to therapy?" - The setup for the joke asks about a seemingly ordinary pizza going to therapy, which is an unusual and unexpected scenario.\n* "Because..." - The phrase "because" sets up the punchline as an explanation or reason for the pizza\'s actions.\n* "...it was feeling a little crusty" - The phrase "a little crusty" has a double meaning. Crusty can refer to the crispy, flaky exterior of a pizza, but it also means being gruff or irritable (e.g., "I\'m feeling a little crusty today"). This is where the wordplay starts.\n* "...and wanted to work through some topping issues!" - The punchline reveals that the pizza\'s motivation for going to therapy is not ju

In [14]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic': 'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy?\n\nBecause it was feeling a little "twisted"!',
 'explanation': 'A classic play on words!\n\nThe joke is a pun, which is a type of wordplay that exploits multiple meanings of a word or phrase. Here\'s how it works:\n\n* The setup for the joke is "Why did the spaghetti go to therapy?" - This implies that something unusual is happening with the spaghetti.\n* The punchline is "Because it was feeling a little \'twisted\'!" - Now, this is where the pun comes in.\n\nIn this case, "twisted" has a double meaning:\n\n1. **Twisted** can refer to the physical state of cooked spaghetti, which is indeed twisted and curly when it\'s not being served.\n2. **Twisted** also means mentally disturbed or unstable (e.g., feeling anxious or upset).\n\nSo, the joke is saying that the spaghetti went to therapy because it was feeling a little "twisted" - both literally (its physical state) and figuratively (mentally distressed). It\'s a clever p